
# River flood forest restoration avoided EADs and Hurricane Melissa damage exposure

This notebook links the river-flood forest restoration avoided-EAD rasters from DPhil paper 2 with Hurricane Melissa NDVI damage, wind-swath and storm-track exposure layers used in DPhil paper 3.

The analysis is intentionally read-only with respect to DPhil paper 2. It reads the published river-flood restoration avoided-EAD rasters and writes derived Melissa-threat summaries under `dphil_paper_3/results/threats/hurricane_melissa_damage/river_flood_restoration_eads_hurricane_damage/`.

Restoration areas are represented by pixels with positive avoided EAD in either the minimum or maximum river-flood damage scenario. Because these are restoration-benefit areas rather than mapped existing forest patches, Hurricane Melissa damage is estimated from pre- and post-storm HLS NDVI change in restoration-benefit pixels with valid NDVI and baseline NDVI >= 0.20. Damage is defined as a relative NDVI decline of at least 10%.


In [ ]:

from pathlib import Path
from collections import OrderedDict
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
from rasterio.warp import Resampling, reproject
from scipy import ndimage

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


In [ ]:

def find_project_root(start: Path | None = None) -> Path:
    """Find the repository/project root containing dphil_papers."""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "dphil_papers").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing dphil_papers")


ROOT = find_project_root()
PAPER2 = ROOT / "dphil_papers" / "dphil_paper_2"
PAPER3 = ROOT / "dphil_papers" / "dphil_paper_3"

OUT_DIR = PAPER3 / "results" / "threats" / "hurricane_melissa_damage" / "river_flood_restoration_eads_hurricane_damage"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RIVER_EAD_MIN_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_min.tif"
RIVER_EAD_MAX_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_max.tif"
NDVI_BEFORE_PATH = PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_before_epsg3448_2025-08-21_to_2025-10-21.tif"
NDVI_AFTER_PATH = PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_after_epsg3448_2025-10-29_to_2025-12-29.tif"
TRACK_PATH = PAPER3 / "inputs" / "hurricane_melissa_track_noaa" / "al132025_best_track" / "AL132025_lin.shp"
WIND_SWATH_PATH = PAPER3 / "inputs" / "hurricane_melissa_track_noaa" / "al132025_best_track" / "AL132025_windswath.shp"

J2USD = 1.0 / 150.0
REL_BASELINE_MIN = 0.20
REL_DAMAGE_THRESHOLD = -0.10
DAMAGE_THRESHOLDS = [0.05, 0.10, 0.20, 0.30]
DISTANCE_BINS_KM = [0, 25, 50, 75, 100, 150, 200, 300]
WIND_THRESHOLDS_KT = [34, 50, 64]

for path in [RIVER_EAD_MIN_PATH, RIVER_EAD_MAX_PATH, NDVI_BEFORE_PATH, NDVI_AFTER_PATH, TRACK_PATH, WIND_SWATH_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

ROOT, OUT_DIR



## Load river-flood restoration avoided EAD rasters

The avoided-EAD rasters are from DPhil paper 2 and are stored in JMD. They are converted to USD using the same conversion applied in the river-flood analysis (`J2USD = 1/150`). Pixels with positive avoided EAD in either scenario define the river-flood forest restoration benefit area.


In [ ]:

def read_ead_usd_on_reference(path: Path, ref_profile: dict | None = None) -> tuple[np.ndarray, dict]:
    """Read a JMD avoided-EAD raster, check alignment, and return positive USD values with non-positive pixels set to NaN."""
    with rasterio.open(path) as src:
        profile = {
            "crs": src.crs,
            "transform": src.transform,
            "height": src.height,
            "width": src.width,
            "bounds": src.bounds,
            "res": src.res,
        }
        if ref_profile is not None:
            for key in ["crs", "transform", "height", "width"]:
                if profile[key] != ref_profile[key]:
                    raise ValueError(f"{path.name} is not aligned with the reference raster on {key}")
        arr = src.read(1).astype("float64") * J2USD

    arr[~np.isfinite(arr) | (arr <= 0)] = np.nan
    return arr, profile


river_ead_max_usd, ref = read_ead_usd_on_reference(RIVER_EAD_MAX_PATH)
river_ead_min_usd, _ = read_ead_usd_on_reference(RIVER_EAD_MIN_PATH, ref)

transform = ref["transform"]
crs = ref["crs"]
shape = (ref["height"], ref["width"])
pixel_area_ha = abs(transform.a * transform.e) / 10_000

min_positive = np.isfinite(river_ead_min_usd) & (river_ead_min_usd > 0)
max_positive = np.isfinite(river_ead_max_usd) & (river_ead_max_usd > 0)
benefit_mask = min_positive | max_positive

scenario_arrays = OrderedDict(
    minimum=river_ead_min_usd,
    maximum=river_ead_max_usd,
)
scenario_positive_masks = OrderedDict(
    minimum=min_positive,
    maximum=max_positive,
)
scenario_totals_usd = OrderedDict(
    (name, float(np.nansum(np.where(scenario_positive_masks[name], arr, 0.0))))
    for name, arr in scenario_arrays.items()
)

river_grid_summary = pd.DataFrame([
    {
        "scenario": scenario,
        "positive_pixels": int(scenario_positive_masks[scenario].sum()),
        "positive_area_ha": scenario_positive_masks[scenario].sum() * pixel_area_ha,
        "positive_avoided_ead_usd": scenario_totals_usd[scenario],
        "positive_avoided_ead_usd_million": scenario_totals_usd[scenario] / 1e6,
    }
    for scenario in scenario_arrays
])

river_grid_summary



## Reproject HLS NDVI composites to the restoration-benefit grid

The HLS NDVI rasters use the same CRS as the river-flood rasters but a different extent. The pre- and post-Melissa composites are reprojected to the restoration-benefit grid before computing relative change.


In [ ]:

def reproject_continuous_to_reference(path: Path, ref: dict, dst_nodata=np.nan) -> np.ndarray:
    """Reproject a continuous raster to the river-flood restoration-benefit grid."""
    dest = np.full((ref["height"], ref["width"]), dst_nodata, dtype="float32")
    with rasterio.open(path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=dest,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=ref["transform"],
            dst_crs=ref["crs"],
            dst_nodata=dst_nodata,
            resampling=Resampling.bilinear,
        )
    return dest


ndvi_before = reproject_continuous_to_reference(NDVI_BEFORE_PATH, ref)
ndvi_after = reproject_continuous_to_reference(NDVI_AFTER_PATH, ref)

ndvi_eligible_mask = (
    benefit_mask
    & np.isfinite(ndvi_before)
    & np.isfinite(ndvi_after)
    & (ndvi_before >= REL_BASELINE_MIN)
)

relative_ndvi_change = np.full(shape, np.nan, dtype="float32")
np.divide(
    ndvi_after - ndvi_before,
    ndvi_before,
    out=relative_ndvi_change,
    where=ndvi_eligible_mask,
)

hurricane_damage_mask = ndvi_eligible_mask & (relative_ndvi_change <= REL_DAMAGE_THRESHOLD)

pd.DataFrame([
    {
        "benefit_area_ha": benefit_mask.sum() * pixel_area_ha,
        "ndvi_eligible_area_ha": ndvi_eligible_mask.sum() * pixel_area_ha,
        "pct_benefit_area_ndvi_eligible": ndvi_eligible_mask.sum() / benefit_mask.sum() * 100,
        "damaged_area_ha": hurricane_damage_mask.sum() * pixel_area_ha,
        "pct_benefit_area_damaged": hurricane_damage_mask.sum() / benefit_mask.sum() * 100,
        "pct_ndvi_eligible_area_damaged": hurricane_damage_mask.sum() / ndvi_eligible_mask.sum() * 100,
    }
])



## Rasterise Hurricane Melissa wind zones and distance to track

The NOAA best-track shapefiles use geographic coordinates with an authalic-sphere CRS label. As in the mangrove analysis, geometries with longitude/latitude bounds are treated as EPSG:4326 before reprojection to EPSG:3448.


In [ ]:

def read_noaa_lonlat(path: Path, target_crs) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(path)
    bounds = gdf.total_bounds
    if bounds[0] >= -180 and bounds[2] <= 180 and bounds[1] >= -90 and bounds[3] <= 90:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(target_crs)


def rasterize_geometries(geometries, ref: dict, all_touched: bool = True) -> np.ndarray:
    valid_geoms = [geom for geom in geometries if geom is not None and not geom.is_empty]
    if not valid_geoms:
        return np.zeros((ref["height"], ref["width"]), dtype=bool)
    return rasterize(
        [(geom, 1) for geom in valid_geoms],
        out_shape=(ref["height"], ref["width"]),
        transform=ref["transform"],
        fill=0,
        dtype="uint8",
        all_touched=all_touched,
    ).astype(bool)


track = read_noaa_lonlat(TRACK_PATH, crs)
wind_swath = read_noaa_lonlat(WIND_SWATH_PATH, crs)

wind_cumulative_masks = OrderedDict()
for threshold in WIND_THRESHOLDS_KT:
    wind_cumulative_masks[f">={threshold} kt"] = rasterize_geometries(
        wind_swath.loc[wind_swath["RADII"] >= threshold, "geometry"], ref
    )

wind_zone_masks = OrderedDict([
    (">=64 kt", wind_cumulative_masks[">=64 kt"]),
    (">=50 to <64 kt", wind_cumulative_masks[">=50 kt"] & ~wind_cumulative_masks[">=64 kt"]),
    (">=34 to <50 kt", wind_cumulative_masks[">=34 kt"] & ~wind_cumulative_masks[">=50 kt"]),
    ("<34 kt / outside wind swath", ~wind_cumulative_masks[">=34 kt"]),
])

track_mask = rasterize_geometries(track.geometry, ref)
if not track_mask.any():
    raise ValueError("Storm track did not intersect the river-flood restoration raster grid")

distance_to_track_km = ndimage.distance_transform_edt(
    ~track_mask,
    sampling=(abs(transform.e), abs(transform.a)),
) / 1000.0

distance_bin_masks = OrderedDict()
for lower, upper in zip(DISTANCE_BINS_KM[:-1], DISTANCE_BINS_KM[1:]):
    distance_bin_masks[f"{lower}-{upper} km"] = (distance_to_track_km >= lower) & (distance_to_track_km < upper)

distance_bin_masks[f">={DISTANCE_BINS_KM[-1]} km"] = distance_to_track_km >= DISTANCE_BINS_KM[-1]

pd.DataFrame([
    {"wind_zone": label, "benefit_area_ha": (benefit_mask & mask).sum() * pixel_area_ha}
    for label, mask in wind_zone_masks.items()
])



## Summarise damage exposure and avoided EADs

The summary tables report area, observed NDVI damage, and avoided EADs in each storm-exposure class. The EAD columns distinguish between the total positive avoided EAD in the class, the share of national river-flood restoration avoided EAD in the class, and the avoided EAD located in pixels with observed >10% NDVI decline.


In [ ]:

def pct(numerator: float, denominator: float) -> float:
    if denominator == 0 or not np.isfinite(denominator):
        return np.nan
    return float(numerator / denominator * 100)


def area_ha(mask: np.ndarray) -> float:
    return float(mask.sum() * pixel_area_ha)


def ead_sum_usd(arr: np.ndarray, mask: np.ndarray) -> float:
    return float(np.nansum(np.where(mask & np.isfinite(arr), arr, 0.0)))


def summary_record(label: str, mask: np.ndarray) -> dict:
    benefit = benefit_mask & mask
    eligible = ndvi_eligible_mask & mask
    damaged = hurricane_damage_mask & mask

    rec = {
        "group": label,
        "benefit_pixels": int(benefit.sum()),
        "benefit_area_ha": area_ha(benefit),
        "ndvi_eligible_area_ha": area_ha(eligible),
        "pct_benefit_area_ndvi_eligible": pct(eligible.sum(), benefit.sum()),
        "damaged_area_ha": area_ha(damaged),
        "pct_benefit_area_damaged": pct(damaged.sum(), benefit.sum()),
        "pct_ndvi_eligible_area_damaged": pct(damaged.sum(), eligible.sum()),
    }

    for scenario, arr in scenario_arrays.items():
        positive = scenario_positive_masks[scenario]
        scenario_total = scenario_totals_usd[scenario]
        class_ead = ead_sum_usd(arr, benefit & positive)
        eligible_ead = ead_sum_usd(arr, eligible & positive)
        damaged_ead = ead_sum_usd(arr, damaged & positive)
        rec[f"positive_avoided_ead_usd_{scenario}"] = class_ead
        rec[f"positive_avoided_ead_usd_million_{scenario}"] = class_ead / 1e6
        rec[f"pct_total_positive_avoided_ead_{scenario}"] = pct(class_ead, scenario_total)
        rec[f"ndvi_eligible_positive_avoided_ead_usd_{scenario}"] = eligible_ead
        rec[f"ndvi_eligible_positive_avoided_ead_usd_million_{scenario}"] = eligible_ead / 1e6
        rec[f"pct_class_positive_avoided_ead_ndvi_eligible_{scenario}"] = pct(eligible_ead, class_ead)
        rec[f"damaged_positive_avoided_ead_usd_{scenario}"] = damaged_ead
        rec[f"damaged_positive_avoided_ead_usd_million_{scenario}"] = damaged_ead / 1e6
        rec[f"pct_class_positive_avoided_ead_damaged_{scenario}"] = pct(damaged_ead, class_ead)
        rec[f"pct_total_positive_avoided_ead_damaged_{scenario}"] = pct(damaged_ead, scenario_total)

    return rec


overall_summary = pd.DataFrame([summary_record("all restoration-benefit pixels", np.ones(shape, dtype=bool))])
wind_zone_summary = pd.DataFrame([summary_record(label, mask) for label, mask in wind_zone_masks.items()])
wind_threshold_summary = pd.DataFrame([summary_record(label, mask) for label, mask in wind_cumulative_masks.items()])
distance_bin_summary = pd.DataFrame([summary_record(label, mask) for label, mask in distance_bin_masks.items()])

scenario_summary = pd.DataFrame([
    {
        "scenario": scenario,
        "positive_pixels": int(scenario_positive_masks[scenario].sum()),
        "positive_area_ha": area_ha(scenario_positive_masks[scenario]),
        "positive_avoided_ead_usd": scenario_totals_usd[scenario],
        "positive_avoided_ead_usd_million": scenario_totals_usd[scenario] / 1e6,
        "positive_area_damaged_ha": area_ha(hurricane_damage_mask & scenario_positive_masks[scenario]),
        "damaged_positive_avoided_ead_usd": ead_sum_usd(arr, hurricane_damage_mask & scenario_positive_masks[scenario]),
        "damaged_positive_avoided_ead_usd_million": ead_sum_usd(arr, hurricane_damage_mask & scenario_positive_masks[scenario]) / 1e6,
        "pct_positive_avoided_ead_damaged": pct(
            ead_sum_usd(arr, hurricane_damage_mask & scenario_positive_masks[scenario]),
            scenario_totals_usd[scenario],
        ),
    }
    for scenario, arr in scenario_arrays.items()
])

overall_summary


In [ ]:

wind_zone_summary[[
    "group", "benefit_area_ha", "ndvi_eligible_area_ha", "damaged_area_ha",
    "pct_benefit_area_damaged", "pct_ndvi_eligible_area_damaged",
    "positive_avoided_ead_usd_million_minimum", "positive_avoided_ead_usd_million_maximum",
    "pct_total_positive_avoided_ead_minimum", "pct_total_positive_avoided_ead_maximum",
    "damaged_positive_avoided_ead_usd_million_minimum", "damaged_positive_avoided_ead_usd_million_maximum",
    "pct_total_positive_avoided_ead_damaged_minimum", "pct_total_positive_avoided_ead_damaged_maximum",
]]


In [ ]:

distance_bin_summary[[
    "group", "benefit_area_ha", "ndvi_eligible_area_ha", "damaged_area_ha",
    "pct_benefit_area_damaged", "pct_ndvi_eligible_area_damaged",
    "positive_avoided_ead_usd_million_minimum", "positive_avoided_ead_usd_million_maximum",
    "pct_total_positive_avoided_ead_minimum", "pct_total_positive_avoided_ead_maximum",
    "damaged_positive_avoided_ead_usd_million_minimum", "damaged_positive_avoided_ead_usd_million_maximum",
    "pct_total_positive_avoided_ead_damaged_minimum", "pct_total_positive_avoided_ead_damaged_maximum",
]]


In [ ]:

def threshold_record(threshold: float) -> dict:
    damaged = ndvi_eligible_mask & (relative_ndvi_change <= -threshold)
    rec = {
        "relative_ndvi_decline_threshold": threshold,
        "damaged_area_ha": area_ha(damaged),
        "pct_benefit_area_damaged": pct(damaged.sum(), benefit_mask.sum()),
        "pct_ndvi_eligible_area_damaged": pct(damaged.sum(), ndvi_eligible_mask.sum()),
    }
    for scenario, arr in scenario_arrays.items():
        damaged_ead = ead_sum_usd(arr, damaged & scenario_positive_masks[scenario])
        rec[f"damaged_positive_avoided_ead_usd_{scenario}"] = damaged_ead
        rec[f"damaged_positive_avoided_ead_usd_million_{scenario}"] = damaged_ead / 1e6
        rec[f"pct_total_positive_avoided_ead_damaged_{scenario}"] = pct(damaged_ead, scenario_totals_usd[scenario])
    return rec


damage_threshold_summary = pd.DataFrame([threshold_record(threshold) for threshold in DAMAGE_THRESHOLDS])
damage_threshold_summary



## Write output tables and summary text


In [ ]:

def write_csv(df: pd.DataFrame, filename: str) -> Path:
    path = OUT_DIR / filename
    df.to_csv(path, index=False)
    return path


written_paths = []
written_paths.append(write_csv(river_grid_summary, "river_flood_restoration_ead_hurricane_damage_grid_summary.csv"))
written_paths.append(write_csv(overall_summary, "river_flood_restoration_ead_hurricane_damage_overall_summary.csv"))
written_paths.append(write_csv(scenario_summary, "river_flood_restoration_ead_hurricane_damage_scenario_summary.csv"))
written_paths.append(write_csv(wind_zone_summary, "river_flood_restoration_ead_hurricane_damage_by_wind_zone_exclusive.csv"))
written_paths.append(write_csv(wind_threshold_summary, "river_flood_restoration_ead_hurricane_damage_by_wind_threshold_cumulative.csv"))
written_paths.append(write_csv(distance_bin_summary, "river_flood_restoration_ead_hurricane_damage_by_distance_bin.csv"))
written_paths.append(write_csv(damage_threshold_summary, "river_flood_restoration_ead_hurricane_damage_by_damage_threshold.csv"))

written_paths


In [ ]:
def fnum(value: float, digits: int = 1) -> str:
    return f"{value:,.{digits}f}"


def fpct(value: float, digits: int = 1) -> str:
    if not np.isfinite(value):
        return "NA"
    return f"{value:.{digits}f}%"


def scenario_pct(row: pd.Series, prefix: str) -> str:
    return (
        f"{fpct(row[f'{prefix}_minimum'])} and "
        f"{fpct(row[f'{prefix}_maximum'])} in the minimum and maximum scenarios, respectively"
    )


def usd_million_range(row: pd.Series, prefix: str, digits: int = 2) -> str:
    return f"US${row[f'{prefix}_minimum'] / 1e6:.{digits}f}-{row[f'{prefix}_maximum'] / 1e6:.{digits}f} million"


def get_row(df: pd.DataFrame, label: str) -> pd.Series:
    return df.loc[df["group"] == label].iloc[0]


overall = overall_summary.iloc[0]
wind64 = get_row(wind_zone_summary, ">=64 kt")
wind50 = get_row(wind_zone_summary, ">=50 to <64 kt")
wind34 = get_row(wind_zone_summary, ">=34 to <50 kt")
dist0 = get_row(distance_bin_summary, "0-25 km")
dist25 = get_row(distance_bin_summary, "25-50 km")
dist50 = get_row(distance_bin_summary, "50-75 km")

summary_text = f"""# River flood forest restoration and Hurricane Melissa damage

Across Jamaica's {fnum(overall['benefit_area_ha'])} ha of river-flood forest restoration area with positive avoided EAD, {fnum(overall['ndvi_eligible_area_ha'])} ha ({fpct(overall['pct_benefit_area_ndvi_eligible'])}) had valid pre- and post-storm NDVI and baseline NDVI >= {REL_BASELINE_MIN:.2f}. Hurricane Melissa damage, defined as a relative NDVI decline of at least {abs(REL_DAMAGE_THRESHOLD) * 100:.0f}%, affected {fnum(overall['damaged_area_ha'])} ha, equivalent to {fpct(overall['pct_benefit_area_damaged'])} of the restoration-benefit area and {fpct(overall['pct_ndvi_eligible_area_damaged'])} of the NDVI-eligible area. These damaged pixels account for {usd_million_range(overall, 'damaged_positive_avoided_ead_usd')}, representing {scenario_pct(overall, 'pct_total_positive_avoided_ead_damaged')}.

Damage was strongly related to storm exposure. In the >=64 kt wind zone, {fnum(wind64['damaged_area_ha'])} ha of {fnum(wind64['benefit_area_ha'])} ha of restoration-benefit area were damaged ({fpct(wind64['pct_benefit_area_damaged'])}; {fpct(wind64['pct_ndvi_eligible_area_damaged'])} of NDVI-eligible area), compared with {fnum(wind50['damaged_area_ha'])} ha of {fnum(wind50['benefit_area_ha'])} ha ({fpct(wind50['pct_benefit_area_damaged'])}; {fpct(wind50['pct_ndvi_eligible_area_damaged'])} of NDVI-eligible area) in the exclusive >=50 kt zone and {fnum(wind34['damaged_area_ha'])} ha of {fnum(wind34['benefit_area_ha'])} ha ({fpct(wind34['pct_benefit_area_damaged'])}; {fpct(wind34['pct_ndvi_eligible_area_damaged'])} of NDVI-eligible area) in the exclusive >=34 kt zone. Similarly, restoration-benefit areas within 25 km of the storm track had {fnum(dist0['damaged_area_ha'])} ha of {fnum(dist0['benefit_area_ha'])} ha damaged ({fpct(dist0['pct_benefit_area_damaged'])}; {fpct(dist0['pct_ndvi_eligible_area_damaged'])} of NDVI-eligible area), compared with {fnum(dist25['damaged_area_ha'])} ha of {fnum(dist25['benefit_area_ha'])} ha ({fpct(dist25['pct_benefit_area_damaged'])}; {fpct(dist25['pct_ndvi_eligible_area_damaged'])} of NDVI-eligible area) in the 25-50 km distance band and {fnum(dist50['damaged_area_ha'])} ha of {fnum(dist50['benefit_area_ha'])} ha ({fpct(dist50['pct_benefit_area_damaged'])}; {fpct(dist50['pct_ndvi_eligible_area_damaged'])} of NDVI-eligible area) in the 50-75 km band.

The highest-wind restoration-benefit areas also account for a large share of estimated river-flood risk reduction: pixels in the >=64 kt wind zone account for {usd_million_range(wind64, 'positive_avoided_ead_usd')}, representing {scenario_pct(wind64, 'pct_total_positive_avoided_ead')}. By contrast, restoration-benefit areas within 25 km of the track account for {usd_million_range(dist0, 'positive_avoided_ead_usd')}, representing {scenario_pct(dist0, 'pct_total_positive_avoided_ead')}. This indicates that Hurricane Melissa's strongest NDVI damage signal overlaps materially with river-flood restoration-benefit areas, particularly in the >=64 kt wind zone, although a substantial share of avoided EAD is also located in lower-damage areas farther from the storm track.
"""

summary_path = OUT_DIR / "river_flood_restoration_ead_hurricane_damage_written_summary.md"
summary_path.write_text(summary_text)
print(summary_text)
print("\nWrote: " + str(summary_path))